In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

# QUICK=True re-runs a fast, few-seed version of an experiment in-kernel (serial, no Pool);
# the default (QUICK=False) REPLAYS the committed 20-seed grid so every figure renders
# instantly. The dopamine/EEG capstones REQUIRE external data absent here, so they are
# always replay-only (a `--full` shell command is documented instead); only the Frank task's
# synthetic-reward device conditions can be recomputed in-kernel.
QUICK = False

GREEN, INDIGO, RED, GOLD, GREY = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2"

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

def _running(v, w=50):
    v = np.asarray(v, float)
    c = np.cumsum(np.insert(v, 0, 0.0))
    return (c[w:] - c[:-w]) / w

print("data/results:", paths.results_dir())

In [ ]:
r = paths.load_result("exp11_dopamine_capstone.npy")   # external DANDI cache absent -> replay only
chance, crit = float(r["chance"]), float(r["crit"])
meta = r["meta"]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.4, 3.8),
                               gridspec_kw={"width_ratios": [1.0, 1.4]})

# (a) G-shallow: single-layer final reward rate by reward source, bootstrap 95% CI
sh = r["shallow"]
snames = ["synthetic", "dopamine", "shuffled"]; scols = [GREEN, GOLD, GREY]
sm = [np.asarray(sh["finals"][n]).mean() for n in snames]
sci = [sh["ci"][n] for n in snames]
serr = [[m - lo for m, (lo, hi) in zip(sm, sci)], [hi - m for m, (lo, hi) in zip(sm, sci)]]
axA.bar(range(3), sm, 0.62, color=scols, yerr=serr, capsize=3, edgecolor="white")
axA.axhline(crit, ls=":", color="0.3", lw=1.0); axA.axhline(chance, ls="--", color="0.3", lw=1.0)
axA.set_xticks(range(3)); axA.set_xticklabels(["syn", "DA", "shuf"], fontsize=9)
axA.set_ylabel("final reward rate"); axA.set_ylim(0, 1.05)
axA.set_title("(a) G-shallow | DA bal-acc {:.2f}".format(meta["mean_bal_acc"]),
              loc="left", fontsize=9.5, fontweight="bold"); _clean(axA)

# (b) G-deep: DFA vs DFA+homeostasis x reward source
dp = r["deep"]
order = [("dfa", "synthetic"), ("dfa", "dopamine"), ("dfa", "shuffled"),
         ("dfa_homeo", "synthetic"), ("dfa_homeo", "dopamine"), ("dfa_homeo", "shuffled")]
dcol = {"synthetic": GREEN, "dopamine": GOLD, "shuffled": GREY}
keys = [str(t) for t in order]
dm = [np.asarray(dp["finals"][k]).mean() for k in keys]
dci = [dp["ci"][k] for k in keys]
derr = [[m - lo for m, (lo, hi) in zip(dm, dci)], [hi - m for m, (lo, hi) in zip(dm, dci)]]
x = np.arange(6)
axB.bar(x, dm, 0.7, color=[dcol[rw] for (_, rw) in order], yerr=derr, capsize=3, edgecolor="white")
axB.axhline(crit, ls=":", color="0.3", lw=1.0); axB.axhline(chance, ls="--", color="0.3", lw=1.0)
axB.axvline(2.5, color="0.8", lw=0.8)
axB.set_xticks(x); axB.set_xticklabels(["syn", "DA", "shuf", "syn", "DA", "shuf"], fontsize=8)
axB.set_ylabel("final reward rate"); axB.set_ylim(0, 1.1)
axB.text(1.0, 1.05, "DFA", ha="center", fontsize=8.5)
axB.text(4.0, 1.05, "DFA + homeostasis", ha="center", fontsize=8.5)
axB.set_title("(b) G-deep | homeostasis under measured dopamine",
              loc="left", fontsize=9.5, fontweight="bold"); _clean(axB)
plt.show()

cr = r["criteria"]
print("pre-registered criteria:", {k: ("PASS" if v else "fail") for k, v in cr.items()})
print(f"decoder: n_subj={meta['n_subj']}  bal-acc={meta['mean_bal_acc']:.3f}  AUC={meta['mean_auc']:.3f}"
      f"  P(dec=R|reward)={meta['P_R1_given_reward']:.3f}  P(dec=R|omission)={meta['P_R1_given_omission']:.3f}")
print(f"G4 (descriptive): DA deep DFA+homeo {dp['finals'][str(('dfa_homeo','dopamine'))].mean():.3f}"
      f"  shallow {sh['finals']['dopamine'].mean():.3f}  (cf. EEG deep 0.62 / shallow 0.66)")

In [ ]:
r = paths.load_result("exp9_capstone.npy")   # external ds003474 EEG absent -> replay only
chance, crit = float(r["chance"]), float(r["crit"])
meta = r["meta"]

rules = ["dfa", "dfa_homeo"]; rewards = ["synthetic", "eeg", "shuffled"]
rcol = {"synthetic": GREEN, "eeg": RED, "shuffled": GREY}
rlab = {"synthetic": "syn", "eeg": "EEG", "shuffled": "shuf"}

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.6, 3.9),
                               gridspec_kw={"width_ratios": [1.5, 1.2]})

# (a) learning curves, DFA+homeostasis by reward source (the composed deep agent)
for rw in rewards:
    cur = np.asarray(r["curves"][str(("dfa_homeo", rw))], float)
    axA.plot(_running(cur), color=rcol[rw], lw=1.7, label=f"DFA+homeo | {rlab[rw]}")
cur_df = np.asarray(r["curves"][str(("dfa", "eeg"))], float)
axA.plot(_running(cur_df), color=INDIGO, lw=1.4, ls="--", label="DFA (no homeo) | EEG")
axA.axhline(crit, ls=":", color="0.3", lw=1.0); axA.axhline(chance, ls="--", color="0.3", lw=1.0)
axA.set_xlabel("trial"); axA.set_ylabel("reward rate (task)"); axA.set_ylim(0.3, 1.05)
axA.set_title("(a) deep XOR under EEG reward", loc="left", fontsize=9.5, fontweight="bold")
axA.legend(frameon=False, fontsize=7.5, loc="lower right"); _clean(axA)

# (b) final reward rate: 2 rules x 3 reward sources, bootstrap 95% CI
order = [(ru, rw) for ru in rules for rw in rewards]
keys = [str(t) for t in order]
m = [np.asarray(r["finals"][k]).mean() for k in keys]
ci = [r["ci"][k] for k in keys]
err = [[mm - lo for mm, (lo, hi) in zip(m, ci)], [hi - mm for mm, (lo, hi) in zip(m, ci)]]
x = np.arange(6)
axB.bar(x, m, 0.7, color=[rcol[rw] for (_, rw) in order], yerr=err, capsize=3, edgecolor="white")
axB.axhline(crit, ls=":", color="0.3", lw=1.0); axB.axhline(chance, ls="--", color="0.3", lw=1.0)
axB.axvline(2.5, color="0.8", lw=0.8)
axB.set_xticks(x); axB.set_xticklabels([rlab[rw] for (_, rw) in order], fontsize=8)
axB.set_ylabel("final reward rate"); axB.set_ylim(0, 1.1)
axB.text(1.0, 1.05, "DFA", ha="center", fontsize=8.5)
axB.text(4.0, 1.05, "DFA + homeostasis", ha="center", fontsize=8.5)
axB.set_title(f"(b) final rate | EEG bal-acc {meta['mean_bal_acc']:.2f}",
              loc="left", fontsize=9.5, fontweight="bold"); _clean(axB)
plt.show()

cr = r["criteria"]
print("pre-registered criteria:", {k: ("PASS" if v else "fail") for k, v in cr.items()})
print("  E2 = homeostasis-robustness headline; E3 = genuine reward info; E4 = graceful degradation")
print(f"decoder: n_subj={meta['n_subj']}  bal-acc={meta['mean_bal_acc']:.3f}"
      f"  P(R=1|correct)={meta['P_R1_given_correct']:.3f}  P(R=1|incorrect)={meta['P_R1_given_incorrect']:.3f}")
for k in ["('dfa_homeo', 'synthetic')", "('dfa_homeo', 'eeg')", "('dfa_homeo', 'shuffled')",
          "('dfa', 'eeg')"]:
    lo, hi = r["ci"][k]
    print(f"  {k:34s}: {np.asarray(r['finals'][k]).mean():.3f}  CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
r = paths.load_result("exp7_biosignal.npy")   # external ds003474 EEG absent -> replay only
chance, crit = float(r["chance"]), float(r["crit"])
bacc = r["meta"]["mean_bal_acc"]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 3.6),
                               gridspec_kw={"width_ratios": [1.5, 1.0]})

# (a) learning curves by reward source (curves are seed-mean running reward rate)
cur = r["curves"]
tt = np.arange(50, 50 + len(cur["synthetic"]))
axA.plot(tt, cur["synthetic"], color=GREEN, lw=1.8, label="synthetic")
axA.plot(tt, cur["biosignal"], color=RED, lw=1.8, label="real-EEG")
axA.plot(tt, cur["shuffled"], color=GREY, lw=1.6, label="shuffled")
axA.axhline(crit, ls=":", color="0.3", lw=1.0); axA.axhline(chance, ls="--", color="0.3", lw=1.0)
axA.set_xlabel("trial"); axA.set_ylabel("reward rate (task)"); axA.set_ylim(0.3, 1.05)
axA.set_title("(a) single-layer, EEG reward", loc="left", fontsize=9.5, fontweight="bold")
axA.legend(frameon=False, fontsize=8, loc="lower right"); _clean(axA)

# (b) final reward bars with bootstrap CI
names = ["synthetic", "biosignal", "shuffled"]; cols = [GREEN, RED, GREY]
means = [np.asarray(r["finals"][n]).mean() for n in names]
cis = [bootstrap_ci(np.asarray(r["finals"][n])) for n in names]
err = [[m - lo for m, (lo, hi) in zip(means, cis)], [hi - m for m, (lo, hi) in zip(means, cis)]]
axB.bar(range(3), means, 0.62, yerr=err, color=cols, capsize=3, edgecolor="white")
axB.axhline(crit, ls=":", color="0.3", lw=1.0); axB.axhline(chance, ls="--", color="0.3", lw=1.0)
axB.set_xticks(range(3)); axB.set_xticklabels(["syn", "EEG", "shuf"], fontsize=9)
axB.set_ylabel("final reward rate"); axB.set_ylim(0, 1.05)
axB.set_title(f"(b) final rate | bal-acc {bacc:.2f}", loc="left", fontsize=9.5, fontweight="bold")
_clean(axB)
plt.show()

for n in names:
    lo, hi = bootstrap_ci(np.asarray(r["finals"][n]))
    print(f"  {n:10s}: {np.asarray(r['finals'][n]).mean():.3f}  CI [{lo:.3f}, {hi:.3f}]")
print("real-EEG beats shuffled with disjoint CIs -> genuine outcome information at the single layer.")

In [ ]:
r = paths.load_result("exp10_probselect.npy")   # EEG-reward conditions need absent ds003474 pools
chance, crit = float(r["chance"]), float(r["crit"])

if QUICK:
    from mrl_trace.probselect import run_probselect
    for c in ("shallow", "dfa", "dfa_homeo"):   # synthetic reward -> no external data needed
        out = run_probselect(c, trials=1500, seeds=4)
        r["finals"][c] = out["finals"]
        r["ci"][c] = bootstrap_ci(np.asarray(out["finals"]))
        r["tests"][c] = {"chooseA": out["chooseA"], "avoidB": out["avoidB"]}

fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(11.4, 3.7),
                                    gridspec_kw={"width_ratios": [1.2, 1.3, 1.1]})

# (a) train-phase learning curves (running P(better stimulus))
ccol = {"shallow": INDIGO, "dfa": GOLD, "dfa_homeo": GREEN, "no_trace": GREY}
clab = {"shallow": "shallow", "dfa": "DFA", "dfa_homeo": "DFA+homeo", "no_trace": "no-trace"}
for c in ["shallow", "dfa", "dfa_homeo", "no_trace"]:
    cur = np.asarray(r["curves"][c], float)
    axA.plot(_running(cur, 100), color=ccol[c], lw=1.7, label=clab[c])
axA.axhline(crit, ls=":", color="0.3", lw=1.0); axA.axhline(chance, ls="--", color="0.3", lw=1.0)
axA.set_xlabel("trial"); axA.set_ylabel("P(better stimulus)"); axA.set_ylim(0.3, 1.02)
axA.set_title("(a) Frank PST -- train", loc="left", fontsize=9.5, fontweight="bold")
axA.legend(frameon=False, fontsize=7.5, loc="lower right"); _clean(axA)

# (b) final train accuracy by condition (bootstrap 95% CI)
order = ["shallow", "dfa", "dfa_homeo", "no_trace", "dfa_homeo_eeg", "dfa_homeo_shuf"]
blab = {"shallow": "shallow", "dfa": "DFA", "dfa_homeo": "DFA\n+homeo", "no_trace": "no-\ntrace",
        "dfa_homeo_eeg": "EEG\nreward", "dfa_homeo_shuf": "EEG\nshuf"}
bcol = {"shallow": INDIGO, "dfa": GOLD, "dfa_homeo": GREEN, "no_trace": GREY,
        "dfa_homeo_eeg": RED, "dfa_homeo_shuf": GREY}
m = [np.asarray(r["finals"][c]).mean() for c in order]
ci = [r["ci"][c] for c in order]
err = [[mm - lo for mm, (lo, hi) in zip(m, ci)], [hi - mm for mm, (lo, hi) in zip(m, ci)]]
x = np.arange(len(order))
axB.bar(x, m, 0.66, color=[bcol[c] for c in order], yerr=err, capsize=3, edgecolor="white")
axB.axhline(crit, ls=":", color="0.3", lw=1.0); axB.axhline(chance, ls="--", color="0.3", lw=1.0)
axB.set_xticks(x); axB.set_xticklabels([blab[c] for c in order], fontsize=7.5)
axB.set_ylabel("final train accuracy"); axB.set_ylim(0, 1.05)
axB.set_title("(b) final accuracy by condition", loc="left", fontsize=9.5, fontweight="bold")
_clean(axB)

# (c) Frank test signature: choose-A vs avoid-B (device conditions only carry the test)
tconds = ["shallow", "dfa", "dfa_homeo"]
tcol = {"shallow": INDIGO, "dfa": GOLD, "dfa_homeo": GREEN}
xw = np.arange(len(tconds)); w = 0.38
cA = [np.asarray(r["tests"][c]["chooseA"]).mean() for c in tconds]
aB = [np.asarray(r["tests"][c]["avoidB"]).mean() for c in tconds]
cA_e = [np.abs(np.subtract(*bootstrap_ci(np.asarray(r["tests"][c]["chooseA"])))) / 2 for c in tconds]
aB_e = [np.abs(np.subtract(*bootstrap_ci(np.asarray(r["tests"][c]["avoidB"])))) / 2 for c in tconds]
axC.bar(xw - w / 2, cA, w, color=GREEN, yerr=cA_e, capsize=3, label="choose-A (positive)")
axC.bar(xw + w / 2, aB, w, color=INDIGO, yerr=aB_e, capsize=3, label="avoid-B (negative)")
axC.axhline(chance, ls="--", color="0.3", lw=1.0)
axC.set_xticks(xw); axC.set_xticklabels(["shallow", "DFA", "DFA\n+homeo"], fontsize=7.5)
axC.set_ylabel("test-phase accuracy"); axC.set_ylim(0, 1.1)
axC.set_title("(c) Frank asymmetry (F4)", loc="left", fontsize=9.5, fontweight="bold")
axC.legend(frameon=False, fontsize=7.5, loc="lower left"); _clean(axC)
plt.show()

for c in order:
    lo, hi = r["ci"][c]
    print(f"  {c:16s}: {np.asarray(r['finals'][c]).mean():.3f}  CI [{lo:.3f}, {hi:.3f}]")
print("Frank test signature (choose-A / avoid-B):",
      {c: (round(float(np.mean(r['tests'][c]['chooseA'])), 2),
           round(float(np.mean(r['tests'][c]['avoidB'])), 2)) for c in tconds})
print("F1/F3/F5 PASS; F2 = pre-registered K2 (12-bit code is linearly separable -> shallow solves it).")

In [ ]:
# Uncomment to launch the (external-data-free) Frank sweep as a subprocess (heavy -- minutes):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mrl_trace.probselect", "--quick"])
print("external DANDI/EEG caches are absent here; see the markdown above for the full-scale commands")